# Vaultify R1 Control Panel

This notebook is intentionally **operational only**. Application logic lives under `src/vaultify/`.

**Guardrails**
- Work from `release/v0.1-extraction`.
- Do not copy the old golden notebook cell-by-cell.
- Keep Flask, retrieval, OAuth, MCP, and business logic out of this notebook.
- Run regressions after each bounded extraction step.


In [ ]:
# 1. Install only the current R1 development/test dependencies
%pip install -q flask flask-sqlalchemy flask-login flask-wtf pytest sentence-transformers qdrant-client
print("✅ R1 development dependencies ready.")


In [ ]:
# 2. Clone or update the extracted Vaultify release branch
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/IhabAltekreeti/vaultify.git"
BRANCH = "release/v0.1-extraction"
REPO_DIR = Path("/content/vaultify-r1")

if not (REPO_DIR / ".git").exists():
    subprocess.run([
        "git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print(f"✅ Vaultify ready at {REPO_DIR}")
print(f"✅ Active branch: {BRANCH}")


In [ ]:
# 3. Point Python at the extracted source tree and verify the package
import sys

SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from vaultify import config

assert config.PROJECT_NAME == "Vaultify"
assert config.COLLECTION_NAME == "vaultify_v3_documents"
assert config.VECTOR_SIZE == 384

print("✅ Extracted Vaultify package imports successfully.")
print(f"✅ Collection: {config.COLLECTION_NAME}")


In [ ]:
# 4. Run the current extracted regression suite
import subprocess

result = subprocess.run(
    ["pytest", "-q", "tests/regression"],
    cwd=REPO_DIR,
    env={**os.environ, "PYTHONPATH": SRC_DIR},
)
assert result.returncode == 0, "R1 regression suite failed."
print("✅ R1 extracted regression suite PASS.")


In [ ]:
# 5. Show the compact release state
release_state = (REPO_DIR / "RELEASE_STATE.md").read_text(encoding="utf-8")
print(release_state)


In [ ]:
# 6. Colab-only Qdrant secrets adapter + connection check
from google.colab import userdata
from vaultify.config import COLLECTION_NAME
from vaultify.services.qdrant import create_qdrant_client, list_collection_names

try:
    QDRANT_URL = userdata.get("QDRANT_URL")
    QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add QDRANT_URL and QDRANT_API_KEY to Colab Secrets, then rerun this cell."
    ) from exc

assert QDRANT_URL, "QDRANT_URL is missing from Colab Secrets."
assert QDRANT_API_KEY, "QDRANT_API_KEY is missing from Colab Secrets."

qdrant = create_qdrant_client(url=QDRANT_URL, api_key=QDRANT_API_KEY)
collection_names = list_collection_names(qdrant)

assert COLLECTION_NAME in collection_names, (
    f"Expected collection {COLLECTION_NAME!r} was not found. "
    f"Available collections: {collection_names}"
)

print("✅ QDRANT_URL: Found")
print("✅ QDRANT_API_KEY: Found")
print("✅ Qdrant Cloud connection successful.")
print(f"✅ Expected collection found: {COLLECTION_NAME}")


## Future operational cells

Add cells here only when extracted modules actually exist, for example:
- Groq secrets/runtime adapter,
- full regression command,
- local Flask startup,
- local MCP startup.

Do **not** place implementation code here.
